Resources for Morphology Descriptors
https://www.acm.or.kr/2702-03/#elementor-toc__heading-anchor-7
https://www.cdc.gov/dpdx/az.html
https://parasitology.cvm.ncsu.edu/m_keys/ruminant/terms.html (general terminology)
Electronic Atlas of Parasitology


Calculate class centroid
Calculate cosine similarity from each new image (one you are predicting) to each class centroid


In [ ]:
%pip install open_clip_torch

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
%pip install transformers

In [2]:
import torch
from PIL import Image
import transformers
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load BiomedCLIP from Hugging Face
from open_clip import create_model_from_pretrained, get_tokenizer
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
# ... pre-process image and tokenize labels, then compare using model ...

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

model.eval()

In [4]:
# Image Embedder
def embed_image(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_feature = model.encode_image(inputs)
    image_feature = image_feature / image_feature.norm(dim=-1, keepdim = True)
    
    return image_feature.squeeze().cpu().numpy()

# Class Embedder, but calculate/embed centriod for each class
def embed_class(dataset_path):
    final_embedding = {}
    
    for class_name in os.listdir(dataset_path):
        class_dir = os.path.join(dataset_path, class_name)
        if not os.path.isdir(class_dir):
            continue
        embeddings = []
        
        for img in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img)
            emb = embed_image(img_path)
            embeddings.append(emb)
        embeddings = np.array(embeddings)
        centroid = np.mean(embeddings, axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        
        final_embedding[class_name] = centroid
    return final_embedding

# Predict function
def predict(image_path, final_embed):
    query_img = embed_image(image_path)
    similarity_scores = {}
    
    for class_name, class_emb in final_embed.items():
        score = cosine_similarity([query_img],[class_emb])[0][0]
        similarity_scores[class_name] = score
    final_scores = sorted(similarity_scores.items(), key = lambda x:x[1], reverse = True)
    return final_scores

            

In [7]:
# Create embedded image database from labeled dataset
import pickle

dataset = "C:/AI_Blood_Smear_Research/ParasiteDataSplit/test"

class_embeddings = embed_class(dataset_path = dataset)
class_embeddings_path = "parasite_embeddings1.pkl"
with open(class_embeddings_path, "wb") as f:
    pickle.dump(class_embeddings, f)

In [ ]:

i = 0
single_test_image = "C:/AI_Blood_Smear_Research/NewTest_Images/TestTime/leishmaniatest.png"
test_images_path = "C:/AI_Blood_Smear_Research/NewTest_Images/TestTime"
for im in os.listdir(test_images_path):
    test_image = os.path.join(test_images_path, im)
    with open(class_embeddings_path, "rb") as f:
        class_embeddings = pickle.load(f)
    
    result = predict(image_path=test_image, final_embed=class_embeddings)
    
    # print(result)
    print(f"Image {i}:")
    for parasite, score in result[:3]:
        print(parasite, score)
    print()
    i+=1

Image 0:
Babesia 0.9112706
RBC 0.8936663
Trypanosome 0.87632024

Image 1:
Plasmodium 0.8049705
Babesia 0.7672065
Leukocyte 0.72155166

Image 2:
Leukocyte 0.89943206
Plasmodium 0.86783284
Babesia 0.85030985

Image 3:
Leishmania 0.852234
Trypanosome 0.7928421
Babesia 0.7556025

Image 4:
Leukocyte 0.9332701
Toxoplasma 0.83155394
RBC 0.821939

Image 5:
Toxoplasma 0.7982366
Trypanosome 0.78797984
Babesia 0.785472

Image 6:
Plasmodium 0.9127422
Babesia 0.9029162
RBC 0.8602102

Image 7:
Leishmania 0.8132814
Trypanosome 0.78903013
Babesia 0.6980894

Image 8:
Leishmania 0.7786135
Trypanosome 0.73458135
Babesia 0.66641474

Image 9:
RBC 0.88145226
Babesia 0.8587778
Plasmodium 0.83437943

Image 10:
Plasmodium 0.7247265
Leishmania 0.7112132
Babesia 0.70863366

Image 11:
Babesia 0.8091107
Leukocyte 0.80832225
Plasmodium 0.8079778

Image 12:
Leishmania 0.8138267
Leukocyte 0.78318024
Trypanosome 0.78243107

